# Module 03: Spatial Heterogeneity, Geographically Weighted Regression (GWR) & Multiscale GWR (MGWR)
### *Exploring Spatial Non-Stationarity: Local Weighted Least Squares, Adaptive Bisquare Kernels, Bandwidth Optimization (AICc / CV), Multiscale Process Scales, and Local Parameter Mapping Across 9,308 Wards*

---

## 1. Why Global Models Miss Spatial Heterogeneity

In Modules 01 and 02, we explored **spatial dependence** (autocorrelation) and estimated global spatial autoregressive models (SAR, SEM, SDM). While these models capture geographic spillovers, they still enforce the assumption of **spatial stationarity**:

$$
y_i = \beta_0 + \sum_{k=1}^p \beta_k x_{ik} + \epsilon_i \implies \beta_k \text{ is identical across every square kilometer of the country}
$$

### 1.1 The Reality of Spatial Non-Stationarity
In human geography, public health, and economics, the structural relationship between predictors and outcomes **varies across space**:
- In remote, infrastructure-sparse rural savannahs, establishing a primary health clinic dramatically lowers disease rates and stimulates local commerce.
- In dense, congested metropolitan cores, clinics are already physically proximate, and environmental sanitation, clean water drainage, or electricity tariffs become the binding constraint.
- Forcing a single global $\hat{\beta}_{\text{clinic}}$ obscures the areas of acute marginal return and leads to misallocated development capital!

### 1.2 The Methodological Solution: Local Spatial Regression
To model spatial non-stationarity, we deploy **Geographically Weighted Regression (GWR)** and its advanced generalization, **Multiscale Geographically Weighted Regression (MGWR)**.


## 2. Mathematical Formulations of GWR & Local WLS

### 2.1 The GWR Model Specification
Instead of fixed global parameters, GWR estimates location-specific parameters $\beta(u_i, v_i)$ at each geographic coordinate $(u_i, v_i)$:

$$
y_i = \beta_0(u_i, v_i) + \sum_{k=1}^p \beta_k(u_i, v_i) x_{ik} + \epsilon_i, \quad \epsilon_i \sim N(0, \sigma^2)
$$

### 2.2 Local Weighted Least Squares (WLS) Derivation
At each target point $i$ with coordinates $(u_i, v_i)$, the parameter vector $\hat{\beta}(u_i, v_i)$ is solved by minimizing the geographically weighted sum of squared residuals:

$$
\min_{\beta(u_i, v_i)} \sum_{j=1}^n w_{ij} \left( y_j - \beta_0(u_i, v_i) - \sum_{k=1}^p \beta_k(u_i, v_i) x_{jk} \right)^2
$$

In matrix notation, the closed-form analytical solution is:
$$
\hat{\beta}(u_i, v_i) = \left( X' W(u_i, v_i) X \right)^{-1} X' W(u_i, v_i) y
$$

where $W(u_i, v_i) = \text{diag}(w_{i1}, w_{i2}, \dots, w_{in})$ is an $n \times n$ diagonal spatial weight matrix whose elements represent the spatial proximity of all other observations $j$ to target location $i$.


## 3. Spatial Kernels & Bandwidth Selection

The spatial weighting function $w_{ij} = f(d_{ij}, b)$ determines how spatial influence decays as Euclidean distance $d_{ij}$ increases:

### 3.1 Kernel Weighting Functions
1. **Continuous Gaussian Kernel (Fixed Distance):**
   $$w_{ij} = \exp\left( -\frac{1}{2}\left( \frac{d_{ij}}{b} \right)^2 \right)$$
   Uses a constant distance bandwidth $b$ everywhere. Problematic in datasets with variable density: in dense cities, $b$ includes thousands of points; in sparse rural areas, $b$ may contain zero neighbors.

2. **Adaptive Bisquare Kernel (Recommended for Irregular Administrative Wards):**
   $$
   w_{ij} = \begin{cases} 
   \left[ 1 - \left( \frac{d_{ij}}{b_i} \right)^2 \right]^2 & \text{if } d_{ij} < b_i \\ 
   0 & \text{otherwise} 
   \end{cases}
   $$
   Here, $b_i$ is an **adaptive distance bandwidth** equal to the distance to the $k$-th nearest neighbor of unit $i$. In dense urban wards, the kernel automatically contracts; in sparse rural wards, the kernel expands to guarantee sufficient sample degrees of freedom!

### 3.2 Bandwidth Optimization via AICc & Cross-Validation (CV)
The optimal bandwidth $b$ (or number of neighbors $k$) balances the classical bias-variance trade-off:
- **Too small bandwidth ($b \to 0$):** High local variance, noisy estimates, and severe overfitting.
- **Too large bandwidth ($b \to \infty$):** Low variance, but approaches the global OLS model (underfitting spatial variation).

The optimal bandwidth is found by golden section search minimizing the **Corrected Akaike Information Criterion (AICc)**:
$$
\text{AICc} = 2n \ln(\hat{\sigma}) + n \ln(2\pi) + n \left( \frac{n + \text{tr}(S)}{n - 2 - \text{tr}(S)} \right)
$$
where $S$ is the GWR hat matrix ($ \hat{y} = Sy $), and $\text{tr}(S)$ represents the effective degrees of freedom.


## 4. Multiscale GWR (MGWR): Varying Spatial Scales Across Covariates

Standard GWR imposes an unrealistic constraint: **every single predictor is forced to operate at the exact same spatial bandwidth $b$**.

In physical and socio-economic geography, processes operate across **multiple spatial scales**:
$$
y_i = \beta_{bw_0}(u_i, v_i) + \sum_{k=1}^p \beta_{bw_k}(u_i, v_i) x_{ik} + \epsilon_i
$$

### The Multi-Scale Spectrum:
- **Micro-Scale (Local Process, e.g. $k=40-80$ neighbors):** Highly localized phenomena (e.g. corner market retail competition, localized storm flood risk, crime hotspots).
- **Meso-Scale (Regional Process, e.g. $k=300-600$ neighbors):** Regional phenomena (e.g. state ministry clinic subsidies, river basin malaria ecology).
- **Macro-Scale (Global Process, $k \to n$):** Broad socio-economic phenomena (e.g. federal monetary inflation, currency devaluation).

MGWR estimates variable-specific bandwidths $bw_k$ using an iterative **backfitting algorithm** that converges when parameter estimates stabilize across all spatial scales.


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import libpysal
from scipy.spatial.distance import cdist

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')


In [ ]:
DATA_PATH = '../data/processed/nigeria_wards_master.parquet'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/nigeria_wards_master.parquet'

gdf = gpd.read_parquet(DATA_PATH)
gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
gdf.reset_index(drop=True, inplace=True)

gdf['rwi_mean'] = gdf['rwi_mean'].fillna(gdf['rwi_mean'].median())
gdf['pop_2025_sum'] = gdf['pop_2025_sum'].fillna(gdf['pop_2025_sum'].median())
gdf['health_rate'] = (gdf['health_facilities_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['market_rate'] = (gdf['markets_count'] / (gdf['pop_2025_sum'] + 100)) * 10000
gdf['water_rate'] = (gdf['water_points_count'] / (gdf['pop_2025_sum'] + 100)) * 10000

# Extract centroids and coordinates
centroids = gdf.geometry.centroid
gdf['coord_x'] = centroids.x
gdf['coord_y'] = centroids.y

print(f"Loaded {len(gdf):,} wards with spatial coordinates.")


## 5. Simulating Local GWR Parameter Estimation via Adaptive Bisquare Kernel

To demonstrate the mathematical mechanics of GWR, we implement the **Local Weighted Least Squares** algorithm using an adaptive bisquare kernel ($k=128$ nearest neighbors):


In [ ]:
coords = np.column_stack([gdf['coord_x'].values, gdf['coord_y'].values])
y_vec = gdf['rwi_mean'].values
X_mat = np.column_stack([
    np.ones(len(gdf)),
    gdf['market_rate'].values,
    gdf['health_rate'].values,
    gdf['water_rate'].values
])

# For pedagogical demonstration, evaluate GWR on a stratified spatial sample of 250 wards
sample_indices = np.linspace(0, len(gdf) - 1, 250, dtype=int)
k_bandwidth = 128

local_betas = []
local_r2 = []

for idx in sample_indices:
    target_coord = coords[idx].reshape(1, -1)
    dists = cdist(target_coord, coords).flatten()
    
    # Adaptive bandwidth: distance to k-th nearest neighbor
    sort_dists = np.sort(dists)
    b_adaptive = sort_dists[k_bandwidth]
    
    # Adaptive bisquare weights
    w_i = np.where(dists < b_adaptive, (1.0 - (dists / b_adaptive) ** 2) ** 2, 0.0)
    W_diag = np.diag(w_i)
    
    # Local WLS: beta_i = (X' W X)^(-1) X' W y
    XtW = X_mat.T * w_i
    XtWX = XtW.dot(X_mat)
    XtWy = XtW.dot(y_vec)
    
    try:
        beta_i = np.linalg.solve(XtWX + np.eye(X_mat.shape[1]) * 1e-6, XtWy)
        local_betas.append(beta_i)
        
        # Local R2
        y_pred_local = X_mat.dot(beta_i)
        y_w_mean = np.sum(w_i * y_vec) / np.sum(w_i)
        ss_tot = np.sum(w_i * (y_vec - y_w_mean) ** 2)
        ss_res = np.sum(w_i * (y_vec - y_pred_local) ** 2)
        r2_i = 1.0 - (ss_res / (ss_tot + 1e-8))
        local_r2.append(np.clip(r2_i, 0.0, 1.0))
    except Exception:
        local_betas.append(np.zeros(X_mat.shape[1]))
        local_r2.append(0.0)

local_betas = np.array(local_betas)
sample_gdf = gdf.iloc[sample_indices].copy()
sample_gdf['beta_market'] = local_betas[:, 1]
sample_gdf['beta_health'] = local_betas[:, 2]
sample_gdf['local_r2'] = local_r2

print("=== LOCAL GWR PARAMETER ESTIMATES (SAMPLE SUMMARY) ===")
param_summary = pd.DataFrame({
    'Parameter': ['Intercept (beta_0)', 'Market Rate (beta_1)', 'Health Rate (beta_2)', 'Water Rate (beta_3)'],
    'Min': local_betas.min(axis=0),
    'Median': np.median(local_betas, axis=0),
    'Max': local_betas.max(axis=0),
    'Std Dev': local_betas.std(axis=0)
})
print(param_summary.round(4))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sample_gdf.plot(column='beta_health', cmap='coolwarm', legend=True, ax=ax1,
                legend_kwds={'label': 'Local Beta: Health Facilities Impact on Wealth', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax1.set_title("A. Spatial Non-Stationarity: Local Health Clinic Impact", fontsize=12, fontweight='bold')
ax1.axis('off')

sample_gdf.plot(column='local_r2', cmap='YlGnBu', legend=True, ax=ax2,
                legend_kwds={'label': 'Local R-Squared (Explanatory Power)', 'orientation': 'horizontal', 'shrink': 0.7, 'pad': 0.05})
ax2.set_title("B. Local Explanatory Power (Local R² Surface)", fontsize=12, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()


## 6. Strategic Executive Synthesis & Comparative Model Decision Table

| Model Specification | Mathematical Formulation | Spatial Parameter & Range | What It Assumes | Primary Real-World Application |
| :--- | :--- | :--- | :--- | :--- |
| **OLS (Classical)** | $y = X\beta + \epsilon$ | Aspatial ($\beta \in \mathbb{R}$) | Spatial independence ($\text{Cov}=0$) | Non-spatial baseline; valid only when residual Moran's $I \approx 0$. |
| **Spatial Lag (SAR)** | $y = \rho Wy + X\beta + \epsilon$ | $\rho \in (-1, 1)$, $\text{Mult} = \frac{1}{1-\rho}$ | Behavioral feedback & direct peer spillover | Retail customer catchment pull, disease epidemic diffusion. |
| **Spatial Error (SEM)**| $y = X\beta + u, \; u = \lambda Wu + \epsilon$ | $\lambda \in (-1, 1)$ | Spatially clustered unmeasured shocks | Regional soil chemistry, climate shocks, shared electric grids. |
| **Spatial Durbin (SDM)**| $y = \rho Wy + X\beta + WX\gamma + \epsilon$ | $\rho, \gamma \in (-1, 1)$ | Endogenous and contextual spillovers | Cross-boundary hospital or school investments boosting local welfare. |
| **GWR (Local)** | $y_i = \beta_0(u_i,v_i) + \sum \beta_k(u_i,v_i)x_{ik} + \epsilon_i$ | Local $\hat{\beta}_k(u_i,v_i)$, Bandwidth $b$ | Spatial non-stationarity across regions | Targeting regional fertilizer subsidies or custom health programs. |
| **MGWR (Multiscale)** | $y_i = \sum \beta_{bw_k}(u_i,v_i)x_{ik} + \epsilon_i$ | Variable-specific bandwidths $bw_k$ | Multi-scale spatial processes | Disentangling local clinic impacts from regional climate drivers. |


## Primary Data Sources & Key References

### Primary Geospatial Data Sources
- **Administrative Ward Boundaries:** GRID3 Nigeria Admin-3 Wards (9,308 polygons): [https://grid3.gov.ng/datasets/nigeria/administrative-boundaries](https://grid3.gov.ng/datasets/nigeria/administrative-boundaries)
- **Relative Wealth Index (RWI):** Meta AI Research & UC Berkeley micro-wealth estimates: [https://data.humdata.org/dataset/relative-wealth-index](https://data.humdata.org/dataset/relative-wealth-index)
- **Demographic Population Counts:** WorldPop 2025 Gridded Population Projections: [https://hub.worldpop.org/geodata/listing?id=29](https://hub.worldpop.org/geodata/listing?id=29)
- **Points of Interest Registries:** GRID3 Nigeria Health Clinics, Markets, Water Points, Police, Religious Centers: [https://grid3.gov.ng/datasets](https://grid3.gov.ng/datasets)
- **Disease Epidemiology:** Malaria Atlas Project (MAP) Plasmodium falciparum $Pf\text{PR}_{2-10}$: [https://malariaatlas.org/](https://malariaatlas.org/)
- **Electoral Infrastructure:** INEC Polling Units Location Registry: [https://irev.inecnigeria.org](https://irev.inecnigeria.org)

### Methodological References & Literature
1. **Anselin, L. (1988).** *Spatial Econometrics: Methods and Models*. Kluwer Academic Publishers.
2. **Anselin, L. (1995).** Local Indicators of Spatial Association -- LISA. *Geographical Analysis*, 27(2), 93-115.
3. **Brunsdon, C., Fotheringham, A. S., & Charlton, M. E. (1996).** Geographically weighted regression: a method for exploring spatial nonstationarity. *Geographical Analysis*, 28(4), 281-298.
4. **Fotheringham, A. S., Yang, W., & Kang, W. (2017).** Multiscale geographically weighted regression (MGWR). *Annals of the American Association of Geographers*, 107(6), 1247-1265.
5. **Chi, G., Fang, H., Chatterjee, S., & Blumenstock, J. E. (2022).** Micro-estimate of wealth for all low- and middle-income countries. *PNAS*, 119(3), e2113658119.
6. **Rey, S. J., & Anselin, L. (2007).** PySAL: A Python library for spatial analytical methods. *The Review of Regional Studies*, 37(1), 5-27.
7. **Tobler, W. R. (1970).** A computer movie simulating urban growth in the Detroit region. *Economic Geography*, 46(sup1), 234-240.
8. **Weiss, D. J., et al. (2019).** Mapping the global prevalence, incidence, and mortality of Plasmodium falciparum, 2000-17. *The Lancet*, 394(10195), 322-331.
